###Load Model & Dataset

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

dataset = load_dataset("imdb")

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


###Tokenization

In [2]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64   # reduced from 128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

###Use VERY SMALL DATASET

In [3]:
train_data = tokenized_dataset["train"].shuffle(seed=42).select(range(1000))
eval_data = tokenized_dataset["test"].shuffle(seed=42).select(range(200))


###Data Collator

In [4]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


###FAST TrainingArguments

In [5]:
!pip install --upgrade transformers

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    max_steps=500,                 # stops early
    per_device_train_batch_size=4, # smaller batch
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    #evaluation_strategy="steps", # Removed: Not supported by installed transformers version
    eval_steps=250,
    logging_steps=50,
    save_steps=500,
    fp16=True,
    report_to="none"
)

###Trainer

In [7]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator
)


In [8]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,4.158518
100,4.031184
150,4.091175
200,4.026327
250,4.036040
300,3.710350
350,3.702539
400,3.745778
450,3.738970
500,3.746019


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=3.898690124511719, metrics={'train_runtime': 1725.9798, 'train_samples_per_second': 1.159, 'train_steps_per_second': 0.29, 'total_flos': 32662093824000.0, 'train_loss': 3.898690124511719, 'epoch': 2.0})

### Evaluate

In [9]:
eval_results = trainer.evaluate()
print(eval_results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 3.9263341426849365, 'eval_runtime': 52.3249, 'eval_samples_per_second': 3.822, 'eval_steps_per_second': 0.956, 'epoch': 2.0}


### Perplexity


In [10]:
import math
print("Perplexity:", math.exp(eval_results["eval_loss"]))


Perplexity: 50.72070159209099
